# Notebook 03 · Clasificación binaria con TensorFlow + Keras

| R1-A2-S4 · Redes Neuronales II | 604013/DEEP LEARNING -CONCEPTOS/FU/CAD2202023205/EIAIIPA2026 |
|---|---|
| Universidad de Cundinamarca · Especialización en Inteligencia Artificial | Docente: **Nidia Stella Garcia Roa** |
| Estudiante: **Yohann Javier Nikolay Salazar Cardenas** | Repositorio: `https://github.com/Yohann0011/Redes-Neuronales-II` |


**Problema:** manzana vs banano (clasificación **binaria**) con la misma arquitectura del Notebook 02:
**3 → 4 → 1**. El objetivo **no** es cambiar la lógica matemática, sino comprobar que un *framework*
automatiza exactamente las operaciones que ya se implementaron a mano con NumPy.

> **Alcance:** construcción de un modelo `Sequential`, compilación con optimizador Adam y pérdida
> *binary_crossentropy*, entrenamiento con `fit()`, evaluación con `evaluate()`, inspección de los
> pesos aprendidos y comparación directa con la implementación manual.

---

## 1. ¿Qué automatiza Keras?

| Acción | NumPy (Notebook 02) | TensorFlow + Keras (este notebook) |
|---|---|---|
| Crear capas | Matrices `W` y vectores `b` | `keras.layers.Dense(...)` |
| Activación ReLU | `np.maximum(0, z)` | `activation="relu"` |
| Activación Sigmoide | Función propia | `activation="sigmoid"` |
| Pérdida | `calcular_error()` (BCE manual) | `loss="binary_crossentropy"` |
| Gradientes | `dZ`, `dW`, `db` calculados a mano | Diferenciación automática dentro de `fit()` |
| Actualización de pesos | `W -= lr * dW` | `optimizer` (Adam) |
| Ciclo de entrenamiento | `for epoch in range(...)` | `modelo.fit(epochs=...)` |
| Evaluación | Cálculo manual de *accuracy* | `modelo.evaluate()` |

**Dónde está el *backpropagation* en Keras:** ocurre internamente durante `fit()`. Keras construye un
grafo de computación y usa diferenciación automática (*automatic differentiation*) para obtener los
gradientes de la pérdida respecto a cada parámetro; el optimizador aplica la actualización.


## 2. Entorno de ejecución

En **Google Colab**, TensorFlow viene preinstalado. Se recomienda `Entorno de ejecución → Cambiar tipo
→ GPU` aunque este modelo es tan pequeño que entrena en milisegundos con CPU.

Si el entorno se reinicia, deben volverse a ejecutar **primero** las celdas que crean `X_train`,
`X_test`, `y_train`, `y_test` y `scaler` (sección 3).

In [1]:
import os
# Ambiente silencioso y determinista para TensorFlow (no altera los resultados)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTHONHASHSEED"] = "0"

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.dpi": 80,
    "savefig.dpi": 80,
    "savefig.bbox": None,
    "figure.autolayout": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
})
plt.rcParams["font.family"] = "DejaVu Sans"

import tensorflow as tf
from tensorflow import keras
import random

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("Backend   :", keras.backend.backend())
print("GPU disponible:", bool(tf.config.list_physical_devices('GPU')))

I0000 00:00:1791644414.323464   24058 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1791644417.069815   24058 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.21.0
Keras     : 3.15.1
Backend   : tensorflow
GPU disponible: False


## 3. Datos (idénticos al Notebook 02)

In [2]:
X = np.array([
    # MANZANAS  (clase 0): [peso g, longitud cm, nivel de amarillo 0-1]
    [150, 7.5, 0.10], [160, 8.0, 0.15], [145, 7.2, 0.05],
    [170, 8.3, 0.20], [155, 7.8, 0.10], [165, 8.1, 0.15],
    [140, 7.0, 0.05], [175, 8.5, 0.20], [158, 7.9, 0.10],
    [162, 8.2, 0.15],

    # BANANOS   (clase 1)
    [120, 18.0, 0.90], [125, 19.0, 0.95], [115, 17.5, 0.85],
    [130, 20.0, 1.00], [118, 18.5, 0.90], [123, 19.5, 0.95],
    [110, 17.0, 0.80], [128, 20.5, 1.00], [121, 18.8, 0.90],
    [126, 19.2, 0.95]
], dtype=float)

y = np.array([0] * 10 + [1] * 10)

print("Matriz de características X:", X.shape, "-> (20 frutas, 3 características)")
print("Vector de etiquetas   y:", y.shape, "-> (0 = manzana, 1 = banano)")
print()
print("Primeras 3 manzanas:\n", X[:3])
print("Primeros 3 bananos:\n", X[10:13])


Matriz de características X: (20, 3) -> (20 frutas, 3 características)
Vector de etiquetas   y: (20,) -> (0 = manzana, 1 = banano)

Primeras 3 manzanas:
 [[1.50e+02 7.50e+00 1.00e-01]
 [1.60e+02 8.00e+00 1.50e-01]
 [1.45e+02 7.20e+00 5.00e-02]]
Primeros 3 bananos:
 [[120.    18.     0.9 ]
 [125.    19.     0.95]
 [115.    17.5    0.85]]


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 80 % entrenamiento / 20 % prueba, conservando la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# El escalado se AJUSTA solo con entrenamiento y se APLICA a prueba
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)
print("Distribución de clases en prueba:", np.bincount(y_test))
print()
print("Media de cada característica en entrenamiento (debe ser ~0):")
print(np.round(X_train.mean(axis=0), 6))
print("Desviación estándar (debe ser ~1):")
print(np.round(X_train.std(axis=0), 6))


X_train: (16, 3) | X_test: (4, 3)
y_train: (16,) | y_test: (4,)
Distribución de clases en prueba: [2 2]

Media de cada característica en entrenamiento (debe ser ~0):
[0. 0. 0.]
Desviación estándar (debe ser ~1):
[1. 1. 1.]


In [4]:
# Keras espera las etiquetas de clasificación binaria como matriz columna (m, 1)
y_train_k = y_train.reshape(-1, 1).astype("float32")
y_test_k = y_test.reshape(-1, 1).astype("float32")

print("X_train:", X_train.shape, X_train.dtype)
print("y_train:", y_train_k.shape, y_train_k.dtype)
print("X_test :", X_test.shape, "| y_test:", y_test_k.shape)

X_train: (16, 3) float64
y_train: (16, 1) float32
X_test : (4, 3) | y_test: (4, 1)


## 4. Construcción del modelo

`keras.Sequential` apila capas de forma lineal. `Input(shape=(3,))` declara que cada ejemplo tiene tres
características; `Dense(4, activation="relu")` crea la capa oculta con cuatro neuronas;
`Dense(1, activation="sigmoid")` crea la neurona de salida binaria.

**Regla de escritura:** usar siempre `keras.layers.Dense(...)`. Escribir `Dense(...)` sin importarlo
provoca `NameError: name 'Dense' is not defined`.

In [5]:
tf.random.set_seed(42)

modelo = keras.Sequential([
    keras.layers.Input(shape=(3,)),
    keras.layers.Dense(4, activation="relu", name="oculta_relu"),
    keras.layers.Dense(1, activation="sigmoid", name="salida_sigmoid")
])

modelo.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ oculta_relu (Dense)             │ (None, 4)              │            16 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida_sigmoid (Dense)          │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21 (84.00 B)

 Trainable params: 21 (84.00 B)

 Non-trainable params: 0 (0.00 B)

### 4.1 Lectura del resumen de parámetros

| Capa | Forma de salida | Parámetros | Cálculo |
|---|---|---|---|
| `oculta_relu` | (None, 4) | 16 | 3 × 4 pesos + 4 bias |
| `salida_sigmoid` | (None, 1) | 5 | 4 × 1 pesos + 1 bias |
| **Total** | | **21** | Todos entrenables |

Son exactamente los mismos 21 parámetros que se gestionaron manualmente en el Notebook 02 con
`W1 (3×4)`, `b1 (4)`, `W2 (4×1)` y `b2 (1)`.

In [6]:
total_params = modelo.count_params()
print("Parámetros entrenables declarados por Keras:", total_params)
print("Parámetros de la implementación NumPy       :", 3*4 + 4 + 4*1 + 1)
print("Coinciden:", total_params == (3*4 + 4 + 4*1 + 1))

Parámetros entrenables declarados por Keras: 21
Parámetros de la implementación NumPy       : 21
Coinciden: True


## 5. Compilación

Compilar significa elegir **tres** piezas del proceso de aprendizaje:

| Elemento | Valor usado | Qué hace |
|---|---|---|
| `optimizer` | `Adam(lr=0.01)` | Calcula y aplica las actualizaciones de los pesos a partir de los gradientes (variante adaptativa del descenso de gradiente). |
| `loss` | `binary_crossentropy` | Mide el error para **dos** clases; es la misma BCE implementada a mano. |
| `metrics` | `accuracy` | Porcentaje de predicciones correctas; se reporta pero **no** guía el aprendizaje. |

In [7]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Optimizador:", modelo.optimizer.__class__.__name__)
print("Pérdida    :", modelo.loss)
print("Métricas   :", modelo.metrics_names)

Optimizador: Adam
Pérdida    : binary_crossentropy
Métricas   : ['loss', 'compile_metrics']


## 6. Entrenamiento

`fit()` ejecuta, en cada época: *forward propagation* → cálculo de pérdida → *backpropagation*
(automático) → actualización de parámetros por el optimizador. `verbose=0` silencia la barra de
progreso para recuperar el historial de forma programática.

In [8]:
historial = modelo.fit(
    X_train,
    y_train_k,
    epochs=200,
    batch_size=4,
    verbose=0
)

print("Error inicial (época 1)  :", round(historial.history["loss"][0], 6))
print("Error final   (época 200):", round(historial.history["loss"][-1], 6))
print()
print(f"{'Época':>7}{'Loss':>12}{'Accuracy':>11}")
for e in [0, 24, 49, 99, 149, 199]:
    print(f"{e+1:>7}{historial.history['loss'][e]:>12.6f}{historial.history['accuracy'][e]:>11.4f}")

Error inicial (época 1)  : 1.392065
Error final   (época 200): 0.01708

  Época        Loss   Accuracy
      1    1.392065     0.5000
     25    0.205516     1.0000
     50    0.112124     1.0000
    100    0.047774     1.0000
    150    0.026606     1.0000
    200    0.017080     1.0000


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].plot(historial.history["loss"], color="#c0392b", lw=2)
axes[0].set_title("Figura 7. Pérdida durante el entrenamiento (Keras)")
axes[0].set_xlabel("Época"); axes[0].set_ylabel("binary_crossentropy")
axes[0].set_yscale("log")

axes[1].plot(historial.history["accuracy"], color="#2980b9", lw=2)
axes[1].set_title("Figura 8. Accuracy durante el entrenamiento")
axes[1].set_xlabel("Época"); axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0.4, 1.05)

plt.tight_layout(); plt.savefig("fig05_keras_frutas_binario.png", bbox_inches="tight"); plt.show()

## 7. Evaluación sobre datos no vistos

In [10]:
test_loss, test_accuracy = modelo.evaluate(X_test, y_test_k, verbose=0)

print("=== RESULTADOS EN EL CONJUNTO DE PRUEBA ===")
print("Error (binary_crossentropy):", round(float(test_loss), 6))
print("Accuracy                   :", round(float(test_accuracy), 4))
print()

probabilidades = modelo.predict(X_test, verbose=0)
predicciones = (probabilidades >= 0.5).astype(int).ravel()

print(f"{'Fruta':>6}{'P(banano)':>12}{'Predicción':>13}{'Real':>12}{'Resultado':>11}")
for i in range(len(X_test)):
    pred = "BANANO" if predicciones[i] == 1 else "MANZANA"
    real = "BANANO" if y_test[i] == 1 else "MANZANA"
    ok = "ACIERTO" if pred == real else "ERROR"
    print(f"{i+1:>6}{probabilidades[i,0]:>12.6f}{pred:>13}{real:>12}{ok:>11}")

=== RESULTADOS EN EL CONJUNTO DE PRUEBA ===
Error (binary_crossentropy): 0.017033
Accuracy                   : 1.0

 Fruta   P(banano)   Predicción        Real  Resultado
     1    0.033234      MANZANA     MANZANA    ACIERTO
     2    0.033234      MANZANA     MANZANA    ACIERTO
     3    0.999720       BANANO      BANANO    ACIERTO
     4    0.999746       BANANO      BANANO    ACIERTO


## 8. Predicción de una fruta nueva

La observación debe transformarse con **el mismo `scaler`** ajustado durante el entrenamiento.

In [11]:
fruta_nueva = np.array([[124.0, 19.0, 0.94]])
fruta_nueva_escalada = scaler.transform(fruta_nueva)

probabilidad = modelo.predict(fruta_nueva_escalada, verbose=0)
resultado = "BANANO" if probabilidad[0][0] >= 0.5 else "MANZANA"

print("Fruta nueva (peso, longitud, amarillo):", fruta_nueva[0])
print("Valores escalados                     :", np.round(fruta_nueva_escalada[0], 4))
print("Probabilidad de ser banano            :", round(float(probabilidad[0][0]), 6))
print("La red piensa que es                  :", resultado)

Fruta nueva (peso, longitud, amarillo): [124.    19.     0.94]
Valores escalados                     : [-0.9271  0.9614  0.9859]
Probabilidad de ser banano            : 0.999793
La red piensa que es                  : BANANO


## 9. Inspección de los pesos aprendidos y comparación con NumPy

Keras permite extraer los parámetros con `get_weights()`. Se comparan las **predicciones** del modelo
Keras con las del modelo manual del Notebook 02 sobre las mismas frutas: aunque los valores numéricos
de los pesos difieran (distinta inicialización y distinto optimizador), la **decisión** debe coincidir
porque ambos resuelven el mismo problema separable.

In [12]:
W1_k, b1_k = modelo.get_layer("oculta_relu").get_weights()
W2_k, b2_k = modelo.get_layer("salida_sigmoid").get_weights()

print("Pesos de la capa oculta W1 (3x4):")
print(np.round(W1_k, 4))
print("Bias de la capa oculta b1:", np.round(b1_k.ravel(), 4))
print()
print("Pesos de la salida W2 (4x1):", np.round(W2_k.ravel(), 4))
print("Bias de la salida  b2       :", round(float(b2_k.ravel()[0]), 4))
print()
print("Activaciones ReLU de la capa oculta para las frutas de prueba:")


def relu(z):
    return np.maximum(0.0, z)


print(np.round(relu(X_test @ W1_k + b1_k), 4))
print("-> Las neuronas en 0 están inactivas para esa fruta; las positivas aportan evidencia.")

Pesos de la capa oculta W1 (3x4):
[[ 0.2053 -0.9923 -0.9678  0.3109]
 [-0.3664  1.6038  1.5694 -0.1699]
 [ 0.455   1.5742 -0.0654  0.3394]]
Bias de la capa oculta b1: [-0.4214  1.3422  0.7045 -0.4553]

Pesos de la salida W2 (4x1): [0.7238 1.3785 1.4679 0.1543]
Bias de la salida  b2       : -3.3704

Activaciones ReLU de la capa oculta para las frutas de prueba:
[[0.     0.     0.     0.    ]
 [0.     0.     0.     0.    ]
 [0.     5.066  3.1121 0.    ]
 [0.     4.9991 3.2409 0.    ]]
-> Las neuronas en 0 están inactivas para esa fruta; las positivas aportan evidencia.


## 10. Análisis: qué cambió y qué no cambió respecto a NumPy

| Aspecto | NumPy (manual) | TensorFlow + Keras |
|---|---|---|
| Matemática subyacente | Idéntica: $z=Xw+b$, ReLU, Sigmoide, BCE | Idéntica |
| Cálculo de gradientes | Derivadas escritas a mano | Diferenciación automática |
| Regla de actualización | Descenso de gradiente simple | Adam (momento + tasa adaptativa por parámetro) |
| Código necesario | ~40 líneas de ciclo de entrenamiento | 1 llamada a `fit()` |
| Escalabilidad | Inviable para millones de parámetros | Diseñado para modelos grandes y GPU |
| Control del proceso | Total (ideal para aprender) | Parcial (se configura por hiperparámetros) |
| Reproducibilidad | Exacta con semilla de NumPy | Aproximada: Adam e inicializaciones dependen también de `tf.random.set_seed` |

**Conclusión práctica:** el *framework* no reemplaza la comprensión del fundamento. Quien entiende
*forward*, *loss*, *backpropagation* y actualización puede diagnosticar por qué un modelo de Keras no
aprende (tasa de aprendizaje inadecuada, datos sin escalar, pérdida incompatible con la activación de
salida, capas mal conectadas).

## 11. Errores frecuentes encontrados en Colab y su solución

| Error | Causa | Solución |
|---|---|---|
| `NameError: name 'X_train' is not defined` | Se escaló o entrenó antes de crear la partición | Ejecutar `train_test_split` antes de `StandardScaler` y respetar el orden de las celdas |
| `NameError: name 'Dense' is not defined` | Se escribió `Dense(...)` sin importarlo | Usar `keras.layers.Dense(...)` o `from tensorflow.keras.layers import Dense` |
| `SyntaxError: invalid syntax` en `or epoch ...` | Falta la letra `f` de `for` | Escribir `for epoch in range(epochs):` y revisar sangría |
| Pérdida que no baja de 0.69 | Etiquetas con forma incorrecta o datos sin escalar | Verificar `y.reshape(-1,1)` y aplicar `StandardScaler` |
| `WARNING:tensorflow:...` abundantes | Logs por defecto | `os.environ["TF_CPP_MIN_LOG_LEVEL"]="3"` antes de importar TensorFlow |

En Colab una variable existe **solo** después de ejecutar la celda que la crea. Si el entorno se
reinicia, conviene usar `Entorno de ejecución → Ejecutar todo`.

## 12. Conclusiones del notebook

1. Keras automatiza *backpropagation* y la actualización de parámetros, pero exige declarar
   correctamente optimizador, pérdida y métricas en `compile()`.
2. La elección de la pérdida depende del tipo de salida: `binary_crossentropy` con una neurona
   Sigmoide; `sparse_categorical_crossentropy` con salida Softmax de $n$ neuronas (Notebook 04).
3. `model.summary()` y `get_weights()` son herramientas de verificación: permiten comprobar que la
   arquitectura construida coincide con la diseñada.
4. Los resultados obtenidos coinciden con la implementación manual, lo que valida ambos desarrollos.

## 13. Referencias

- Gulli, A., Kapoor, A., Pal, S., & Ciraci, S. (2023). *Deep Learning with TensorFlow and Keras* (3rd ed.). Packt Publishing.
  Repositorio oficial: https://github.com/PacktPublishing/Deep-Learning-with-TensorFlow-and-Keras-3rd-edition
- Trask, A. (2019). *Grokking Deep Learning*. Manning Publications.
- Google (2026). *Google Colaboratory*. https://colab.research.google.com/